# 04 — XGBoost Tuning — Clasificación Multiclase de Incontinencia Urinaria

## Objetivo

Este notebook es el notebook principal de la issue de modelado XGBoost. Parte de las conclusiones del notebook anterior (`03_xgboost_baseline.ipynb`) donde se detectó un problema de *data leakage* y se seleccionó el **Camino B** como feature set válido.

El objetivo aquí es **optimizar los hiperparámetros del modelo** para maximizar el F1-macro en validación, con la restricción explícita de que el overfitting (diferencia entre F1 de entrenamiento y F1 de validación) no supere el **5%**.

## Camino B — feature set utilizado

Se excluyen las variables que definen directamente el target (`ui_esfuerzo_presente`, `ui_urgencia_presente`) pero se conservan los síntomas indirectos que capturan severidad y frecuencia de la incontinencia, sin revelar su tipo:

| Tipo de variable | Ejemplos | Incluida |
|---|---|---|
| Demográfica | edad, IMC, etnia | ✓ |
| Clínica | diabetes, hipertensión, artritis | ✓ |
| Síntoma indirecto | ui_frecuencia, ui_molestia_percibida | ✓ |
| **Leaky** | **ui_esfuerzo_presente, ui_urgencia_presente** | **✗** |

## Estructura del notebook

| Sección | Contenido |
|---|---|
| **1. Setup** | Carga de librerías, datos y feature set |
| **2. Función objetivo** | Definición de la búsqueda con control de overfitting |
| **3. Búsqueda con Optuna** | 100 trials con TPE sampler |
| **4. Verificación** | Gap train-val, F1 en test |
| **5. Reporte final** | Baseline vs tuned, métricas por clase |
| **6. Persistencia** | Guardado del modelo y artefactos |

---
## Sección 1 — Setup

Cargamos los datos del pipeline de preprocesamiento y la configuración compartida de CV. Las variables leaky ya fueron eliminadas en el pipeline — los CSVs solo contienen el feature set válido. El target viene directamente como strings (`mixed`, `none`, `stress`, `urge`), igual que en el resto de modelos del ensemble.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import optuna
import joblib
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Cargamos datos procesados
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train_scaled.csv').squeeze()
X_test  = pd.read_csv('../data/processed/X_test_scaled.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

cv_config = joblib.load('../models/cv_config.pkl')
skf = StratifiedKFold(**cv_config)

CLASS_ORDER = sorted(y_train.unique())
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'Clases: {CLASS_ORDER}')

Shape train: (15540, 21) | test: (1815, 21)
Clases: ['mixed' 'none' 'stress' 'urge']


---
## Sección 2 — Función objetivo de Optuna

Optuna optimiza hiperparámetros de forma secuencial: cada **trial** propone una combinación de valores, la evalúa y usa ese resultado para guiar la siguiente propuesta. El algoritmo que usamos es **TPE (Tree-structured Parzen Estimator)**, que construye un modelo probabilístico de qué regiones del espacio de búsqueda son más prometedoras.

### Control de overfitting

Dentro de cada trial usamos `cross_validate` con `return_train_score=True` para medir simultáneamente el F1 en entrenamiento y en validación. Si el gap supera el 5%, el trial recibe una penalización de `-1.0` y queda descartado automáticamente. Esto asegura que Optuna no solo maximice la métrica de validación, sino que también respete la restricción de generalización.

### Hiperparámetros buscados

| Parámetro | Rango | Escala | Qué controla |
|---|---|---|---|
| `n_estimators` | 100–800 | lineal | Número de árboles |
| `max_depth` | 3–8 | lineal | Profundidad máxima por árbol |
| `learning_rate` | 0.01–0.3 | log | Tasa de aprendizaje |
| `subsample` | 0.6–1.0 | lineal | Fracción de filas por árbol |
| `colsample_bytree` | 0.5–1.0 | lineal | Fracción de columnas por árbol |
| `min_child_weight` | 1–10 | lineal | Mínimo de muestras en cada hoja |
| `gamma` | 0–5 | lineal | Ganancia mínima para hacer un split |
| `reg_alpha` | 1e-8–1.0 | log | Regularización L1 (lasso) |
| `reg_lambda` | 1e-8–10.0 | log | Regularización L2 (ridge) |

In [ ]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'objective':         'multi:softprob',
        'num_class':         4,
        'eval_metric':       'mlogloss',
        'use_label_encoder': False,
        'random_state':      42,
        'n_jobs':            -1,
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    model = xgb.XGBClassifier(**params)

    cv_results = cross_validate(
        pipeline, X_train, y_train,
        cv=skf,
        scoring='f1_macro',
        return_train_score=True,
        n_jobs=-1
    )

    f1_val   = cv_results['test_score'].mean()
    f1_train = cv_results['train_score'].mean()
    gap      = f1_train - f1_val

    # Penalizar si el overfitting supera el 5%
    if gap > 0.05:
        return -1.0

    return f1_val

---
## Sección 3 — Ejecución de la búsqueda

Lanzamos 100 trials. El sampler TPE empieza con exploración aleatoria en los primeros ~25 trials y luego concentra la búsqueda en las regiones más prometedoras. El tiempo estimado es de 5 a 15 minutos dependiendo del hardware.

Al finalizar, mostramos cuántos trials pasaron el filtro de overfitting (gap ≤ 5%) y los mejores hiperparámetros encontrados.

In [5]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=100, show_progress_bar=True)

# Filtrar trials válidos (que no fueron penalizados)
valid_trials = [t for t in study.trials if t.value is not None and t.value > 0]

print(f"\nTrials totales:  {len(study.trials)}")
print(f"Trials válidos:  {len(valid_trials)}  (pasaron filtro overfitting ≤ 5%)")
print(f"\nMejor F1-val:    {study.best_value:.4f}")
print(f"Mejores params:")
for k, v in study.best_params.items():
    print(f"  {k:<22}: {v}")

  0%|          | 0/100 [00:00<?, ?it/s]


Trials totales:  100
Trials válidos:  65  (pasaron filtro overfitting ≤ 5%)

Mejor F1-val:    0.6323
Mejores params:
  n_estimators          : 525
  max_depth             : 7
  learning_rate         : 0.033360549217319735
  subsample             : 0.813152908730266
  colsample_bytree      : 0.6739230717255622
  min_child_weight      : 6
  gamma                 : 3.63947489565862
  reg_alpha             : 0.0015986108509522614
  reg_lambda            : 0.0028453534985103278


---
## Sección 4 — Verificación del overfitting

Entrenamos el modelo con los mejores hiperparámetros y medimos explícitamente tres métricas para confirmar que cumple la restricción:

- **F1 Train (CV):** rendimiento promedio en los folds de entrenamiento
- **F1 Val (CV):** rendimiento promedio en los folds de validación — estimador del rendimiento real
- **F1 Test:** rendimiento sobre el conjunto de test, que no participó en ningún momento del tuning
- **Gap train-val:** debe ser ≤ 0.05 (restricción del proyecto)
- **Gap val-test:** diferencia entre la estimación CV y el rendimiento real en test — nos avisa si hubo optimismo excesivo durante la búsqueda

In [6]:
best_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    **study.best_params
)

cv_final = cross_validate(
    best_model, X_train_b, y_train_enc,
    cv=SKF, scoring='f1_macro',
    return_train_score=True
)

f1_train_final = cv_final['train_score'].mean()
f1_val_final   = cv_final['test_score'].mean()
gap_final      = f1_train_final - f1_val_final

# F1 en test (datos nunca vistos)
best_model.fit(X_train_b, y_train_enc)
y_pred_test = best_model.predict(X_test_b)
f1_test     = f1_score(y_test_enc, y_pred_test, average='macro')

print("=" * 45)
print("VERIFICACIÓN DE OVERFITTING — MODELO FINAL")
print("=" * 45)
print(f"F1 Train (CV):   {f1_train_final:.4f}")
print(f"F1 Val   (CV):   {f1_val_final:.4f}")
print(f"F1 Test:         {f1_test:.4f}")
print(f"Gap train-val:   {gap_final:.4f}  {'✓ OK' if gap_final <= 0.05 else '✗ EXCEDE 5%'}")
print(f"Gap val-test:    {abs(f1_val_final - f1_test):.4f}  {'✓ OK' if abs(f1_val_final - f1_test) <= 0.05 else '✗ revisar'}")

c:\Users\Coder\Desktop\IA-P6\proyectos\Proyecto7_Equipo3_Multiclase\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:50:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Coder\Desktop\IA-P6\proyectos\Proyecto7_Equipo3_Multiclase\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:50:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Coder\Desktop\IA-P6\proyectos\Proyecto7_Equipo3_Multiclase\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:50:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Coder\Desktop\IA-P6\proyectos\Proyecto7_Equipo3_Multiclase\.venv\Lib\site

VERIFICACIÓN DE OVERFITTING — MODELO FINAL
F1 Train (CV):   0.6783
F1 Val   (CV):   0.6294
F1 Test:         0.4806
Gap train-val:   0.0489  ✓ OK
Gap val-test:    0.1488  ✗ revisar


---
## Sección 5 — Reporte final: baseline vs modelo tuneado

Comparamos el modelo con parámetros por defecto (Camino B, sin tuning) contra el modelo optimizado. Nos interesa especialmente la mejora en `urge` y `stress`, que son las clases más difíciles de separar cuando no disponemos de las variables leaky.

La tabla de mejora por clase (`↑` / `↓`) muestra si el tuning fue efectivo para cada tipo de incontinencia o si alguna clase empeoró como efecto secundario.

In [7]:
from sklearn.metrics import classification_report

# Baseline Camino B (sin tuning, para comparar)
baseline_b = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4,
    eval_metric='mlogloss', use_label_encoder=False,
    random_state=42, n_jobs=-1
)
baseline_b.fit(X_train_b, y_train_enc)
y_pred_base = baseline_b.predict(X_test_b)

print("BASELINE (Camino B, sin tuning):")
print(classification_report(y_test_enc, y_pred_base, target_names=CLASS_ORDER))

print("\nMODELO TUNED (Camino B, Optuna 100 trials):")
print(classification_report(y_test_enc, y_pred_test, target_names=CLASS_ORDER))

# Delta por clase
f1_base_cls  = f1_score(y_test_enc, y_pred_base, average=None)
f1_tuned_cls = f1_score(y_test_enc, y_pred_test, average=None)

print("MEJORA POR CLASE:")
for cls, base, tuned in zip(CLASS_ORDER, f1_base_cls, f1_tuned_cls):
    delta = tuned - base
    arrow = '↑' if delta > 0 else '↓'
    print(f"  {cls:<10}  {base:.3f} → {tuned:.3f}  {arrow} {abs(delta):.3f}")

c:\Users\Coder\Desktop\IA-P6\proyectos\Proyecto7_Equipo3_Multiclase\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:51:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


BASELINE (Camino B, sin tuning):
              precision    recall  f1-score   support

       mixed       0.51      0.50      0.50       328
        none       0.76      0.87      0.81       972
      stress       0.38      0.31      0.34       317
        urge       0.23      0.14      0.18       198

    accuracy                           0.63      1815
   macro avg       0.47      0.46      0.46      1815
weighted avg       0.59      0.63      0.60      1815


MODELO TUNED (Camino B, Optuna 100 trials):
              precision    recall  f1-score   support

       mixed       0.52      0.51      0.51       328
        none       0.76      0.87      0.81       972
      stress       0.40      0.32      0.36       317
        urge       0.32      0.19      0.24       198

    accuracy                           0.64      1815
   macro avg       0.50      0.47      0.48      1815
weighted avg       0.60      0.64      0.62      1815

MEJORA POR CLASE:
  mixed       0.503 → 0.515  ↑ 0.0

---
## Sección 6 — Persistencia del modelo

Guardamos tres artefactos necesarios para que el modelo pueda ser consumido en producción o por el ensemble:

- **`xgboost.pkl`:** el modelo entrenado con los mejores hiperparámetros, listo para hacer predicciones
- **`optuna_study.pkl`:** el objeto de búsqueda completo de Optuna, que permite reproducir o extender la optimización
- **`feature_cols_caminoB.pkl`:** la lista exacta de columnas usadas en el entrenamiento, necesaria para garantizar que los datos de entrada en producción tengan el mismo orden y las mismas variables

> El `label_encoder.pkl` ya fue guardado en el notebook anterior y se reutiliza aquí sin modificación.

In [8]:
joblib.dump(best_model,  '../models/xgboost.pkl')
joblib.dump(study,       '../models/optuna_study.pkl')

# Guardar también las columnas usadas (necesario para producción)
feature_cols = X_train_b.columns.tolist()
joblib.dump(feature_cols, '../models/feature_cols_caminoB.pkl')

print("Archivos guardados:")
print(f"  ../models/xgboost.pkl            ({best_model.__class__.__name__})")
print(f"  ../models/optuna_study.pkl       ({len(study.trials)} trials)")
print(f"  ../models/feature_cols_caminoB.pkl  ({len(feature_cols)} features)")
print(f"\nF1-macro final en test: {f1_test:.4f}")
print(f"Gap train-val:          {gap_final:.4f}  {'✓ ≤ 5%' if gap_final <= 0.05 else '✗ > 5%'}")

Archivos guardados:
  ../models/xgboost.pkl            (XGBClassifier)
  ../models/optuna_study.pkl       (100 trials)
  ../models/feature_cols_caminoB.pkl  (21 features)

F1-macro final en test: 0.4806
Gap train-val:          0.0489  ✓ ≤ 5%
